# Guided Remediation Walkthrough

This notebook demonstrates how the AI-Support Fabric generates and presents remediation plans for detected findings.

## Remediation Philosophy

The remediation engine follows these principles:

1. **Guided, not Automated**: Provide clear steps rather than automatic fixes
2. **Context-Aware**: Remediation tailored to the specific finding
3. **Risk-Conscious**: Flag high-risk actions requiring approval
4. **Educational**: Explain the "why" behind each step

## Remediation Flow

```
Finding → Remediation Engine → Remediation Plan
                                      ↓
                    [Investigation Steps]
                    [Mitigation Actions]
                    [Verification Steps]
                    [Documentation]
```

In [ ]:
# Setup
import requests
import json
from datetime import datetime

GATEWAY_URL = "http://localhost:8080"

def pretty_print(data):
    """Pretty print JSON data"""
    print(json.dumps(data, indent=2))

def display_remediation_plan(plan):
    """Display remediation plan in readable format"""
    print(f"\n{'='*70}")
    print(f"Plan: {plan['title']}")
    print(f"Plan ID: {plan['plan_id']}")
    print(f"Risk Level: {plan['risk_level']}")
    print(f"Automated: {plan['automated']}")
    print(f"{'='*70}\n")
    
    for step in plan['steps']:
        print(f"Step {step['step']}: {step['action']}")
        print(f"  Description: {step['description']}")
        if step.get('command'):
            print(f"  Command: {step['command']}")
        if step.get('requires_approval'):
            print(f"  ⚠️  REQUIRES APPROVAL")
        if step.get('automated'):
            print(f"  🤖 Automated step")
        print()

## Get Recent Findings and Remediation Plans

In [ ]:
# Fetch findings
response = requests.get(f"{GATEWAY_URL}/api/ai/findings?limit=10")
findings_data = response.json()

print(f"Found {findings_data['count']} findings\n")

# Display summary
for finding in findings_data.get('findings', []):
    print(f"- [{finding['severity']}] {finding['title']}")

In [ ]:
# Fetch remediation plans
response = requests.get(f"{GATEWAY_URL}/api/ai/remediation?limit=10")
remediation_data = response.json()

print(f"Found {remediation_data['count']} remediation plans\n")

plans = remediation_data.get('remediation_plans', [])

if plans:
    print("Available Plans:")
    for i, plan in enumerate(plans, 1):
        print(f"{i}. {plan['title']} (Risk: {plan['risk_level']})")
else:
    print("No remediation plans available yet. Run analysis first.")

## Example 1: Latency Spike Remediation

Let's examine the remediation plan for a latency spike.

In [ ]:
# Find latency spike remediation plan
latency_plan = None
for plan in plans:
    if 'latency' in plan['title'].lower():
        latency_plan = plan
        break

if latency_plan:
    display_remediation_plan(latency_plan)
else:
    print("No latency spike remediation plan found.")
    print("Generate one by running: ./scripts/seed_scenarios.sh scenario_latency_spike")

### Understanding Latency Remediation

The latency remediation follows a diagnostic approach:

1. **Investigate**: Check current system load and metrics
2. **Analyze**: Review APM traces to identify bottlenecks
3. **Diagnose**: Check specific subsystems (database, external services)
4. **Mitigate**: Scale resources if needed (requires approval)
5. **Monitor**: Verify improvement

This approach ensures we understand the root cause before making changes.

## Example 2: Configuration Drift Remediation

Configuration drift is more critical - let's see how it's handled.

In [ ]:
# Find config drift remediation plan
config_plan = None
for plan in plans:
    if 'config' in plan['title'].lower() and 'drift' in plan['title'].lower():
        config_plan = plan
        break

if config_plan:
    display_remediation_plan(config_plan)
else:
    print("No config drift remediation plan found.")
    print("Generate one by running: ./scripts/seed_scenarios.sh scenario_config_drift")

### Understanding Config Drift Remediation

Configuration drift requires careful handling:

1. **Verify**: Confirm the drift and its scope
2. **Authorize**: Check if change was intentional and approved
3. **Assess**: Evaluate security implications
4. **Remediate**: Rollback if unauthorized (requires approval)
5. **Prevent**: Enable monitoring to catch future drift
6. **Document**: Update baseline if authorized

Note that rollback requires approval - this prevents automatic changes to production configs.

## Example 3: Authentication Failure Storm Remediation

Security incidents require immediate but measured response.

In [ ]:
# Find auth failure remediation plan
auth_plan = None
for plan in plans:
    if 'auth' in plan['title'].lower():
        auth_plan = plan
        break

if auth_plan:
    display_remediation_plan(auth_plan)
else:
    print("No auth failure remediation plan found.")
    print("Generate one by running: ./scripts/seed_scenarios.sh scenario_auth_error_storm")

### Understanding Auth Failure Remediation

Security incidents balance speed with caution:

1. **Immediate**: Enable rate limiting (automated, low-risk)
2. **Block**: Block attacking IPs (requires approval)
3. **Protect**: Enable account lockout
4. **Alert**: Notify security team (automated)
5. **Monitor**: Track attack progress (automated)
6. **Strengthen**: Consider MFA enforcement

Some steps are automated (rate limiting, alerts) while others require human approval (IP blocking).

## Remediation Plan Structure

Each remediation plan includes:

### Metadata
- `plan_id`: Unique identifier
- `finding_id`: Links to the original finding
- `title`: Human-readable plan name
- `risk_level`: LOW, MEDIUM, HIGH
- `automated`: Whether the plan can be fully automated
- `created_at`: Timestamp

### Steps
Each step contains:
- `step`: Step number
- `action`: Brief action description
- `description`: Detailed explanation
- `command`: (Optional) Command to execute
- `automated`: Whether step can be automated
- `requires_approval`: Whether human approval needed

## Best Practices

When using remediation plans:

1. **Always Review**: Read the full plan before executing
2. **Understand Impact**: Know what each step does
3. **Approve Carefully**: High-risk steps require careful consideration
4. **Document**: Record actions taken and results
5. **Verify**: Always verify that remediation worked

## Next Steps

1. Try running the scenario scripts and examining the remediation plans
2. Explore the UI dashboard at http://localhost:3000
3. Review the documentation in the `docs/` folder
4. Extend the lab with your own detectors and remediation logic